# 08 — Frequency Model: How Often Do Crashes Happen Per Segment?

The occurrence model in `03` had to invent its own negative examples. There is no
observed "no crash here" record, so random segment/time pairs were sampled and
labelled zero.

That produced an artefact rather than a signal: sampling uniformly across segments
made 48% of negatives `service` or `path` against 2% of positives, so
`highway_simple == "service"` became a near-perfect negative indicator. Roughly
half the apparent performance was network composition, not risk.

This notebook takes the standard road-safety approach instead — model the **count**
of crashes per segment with an exposure offset. The zero counts become data rather
than fabrications. In the literature this is a **Safety Performance Function**, the
core tool of the Highway Safety Manual.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = PROJECT_ROOT / "data" / "processed"

snapped = pd.read_csv(PROC / "berlin_accidents_snapped_to_edges.csv")
edges = pd.read_csv(PROC / "berlin_osm_edge_features.csv")

print(f"{len(snapped):,} snapped crashes | {len(edges):,} directed edges")

37,896 snapped crashes | 441,515 directed edges


In [4]:
# One row per undirected segment. A two-way street appears as two directed edges in
# OSM but is one physical segment, and counting it twice would double both the
# crash count and the length offset.
seg = (
    edges.groupby("pair_id")
    .agg(
        length_m=("edge_length_m", "first"),
        highway_simple=("highway_simple", "first"),
        has_cycleway=("has_cycleway", "max"),
        maxspeed_num=("maxspeed_num", "first"),
    )
    .reset_index()
)

crashes = (
    snapped.groupby("pair_id")
    .agg(crash_count=("pair_id", "size"),
         ksi_count=("is_ksi", "sum"),
         fatal_count=("is_fatal", "sum"),
         truck_count=("is_truck", "sum"),
         near_junction=("near_junction", "mean"))
    .reset_index()
)

seg = seg.merge(crashes, on="pair_id", how="left")
FILL = ["crash_count", "ksi_count", "fatal_count", "truck_count", "near_junction"]
seg[FILL] = seg[FILL].fillna(0)

# A zero length would break the log offset.
seg = seg[seg["length_m"] > 0].copy()

print(f"{len(seg):,} undirected segments")
print(f"crashes assigned: {seg['crash_count'].sum():,.0f} of {len(snapped):,}")
print(f"segments with >=1 crash: {(seg['crash_count'] > 0).mean():.1%}")
print(f"\nmean {seg['crash_count'].mean():.4f} | variance {seg['crash_count'].var():.4f}")
print(f"variance / mean = {seg['crash_count'].var() / seg['crash_count'].mean():.2f}")

238,951 undirected segments
crashes assigned: 37,896 of 37,896
segments with >=1 crash: 8.6%

mean 0.1586 | variance 0.5520
variance / mean = 3.48


### 1.1 The distribution requires a negative binomial

238,951 segments, **91.4% with zero crashes**. Mean 0.1586, variance 0.5520 — a
variance-to-mean ratio of **3.48**.

Poisson regression assumes these are equal. They are not, so NB is required rather
than preferred. This is a data-driven model choice, not a stylistic one.

In [5]:
import warnings

import statsmodels.api as sm
import statsmodels.formula.api as smf

seg["log_len"] = np.log(seg["length_m"])
FORMULA = "crash_count ~ C(highway_simple) + maxspeed_num + has_cycleway"

# Poisson is fitted only to measure overdispersion. A Pearson chi2/df near 1.0
# would mean its equal-variance assumption holds.
pois = smf.glm(FORMULA, data=seg, family=sm.families.Poisson(),
               offset=seg["log_len"]).fit()

# smf.negativebinomial estimates the dispersion parameter jointly and diverged on
# this data — alpha overflowed and the Hessian could not be inverted. Fixing alpha
# and fitting NB as a GLM is the standard alternative, selecting alpha by
# likelihood over a grid. Log-likelihoods are comparable at fixed alpha.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fits = []
    for a in np.linspace(0.5, 12.0, 24):
        m = smf.glm(FORMULA, data=seg,
                    family=sm.families.NegativeBinomial(alpha=a),
                    offset=seg["log_len"]).fit()
        fits.append((a, m.llf, m))

best_alpha, _, nb = max(fits, key=lambda r: r[1])

print(f"Poisson  log-lik {pois.llf:>12,.0f}  AIC {pois.aic:>12,.0f}")
print(f"NB       log-lik {nb.llf:>12,.0f}  AIC {nb.aic:>12,.0f}   alpha = {best_alpha:.2f}")
print(f"\nPearson chi2/df: Poisson {pois.pearson_chi2/pois.df_resid:.2f}"
      f"  ->  NB {nb.pearson_chi2/nb.df_resid:.2f}")
print(f"\nmaxspeed coefficient — Poisson {pois.params['maxspeed_num']:+.4f}, "
      f"NB {nb.params['maxspeed_num']:+.4f}")

Poisson  log-lik      -96,271  AIC      192,580
NB       log-lik      -79,747  AIC      159,532   alpha = 4.00

Pearson chi2/df: Poisson 3.89  ->  NB 2.43

maxspeed coefficient — Poisson -0.0113, NB +0.0060


### 2.1 NB fits substantially better

| | Poisson | NB (α = 4.0) |
|---|---|---|
| Log-likelihood | −96,271 | −79,747 |
| AIC | 192,580 | **159,532** |
| Pearson chi²/df | 3.89 | **2.43** |

AIC falls by 33,048.

**Poisson has the sign of `maxspeed` backwards** — −0.0113 against +0.0060 under
NB. Overdispersion inverted it. That is the concrete gain from fitting the correct
error distribution rather than the convenient one: the Poisson result would have
supported the claim that faster roads carry fewer crashes.

In [6]:
tab = pd.DataFrame({
    "coef": nb.params,
    "rate_ratio": np.exp(nb.params),
    "ci_low": np.exp(nb.conf_int()[0]),
    "ci_high": np.exp(nb.conf_int()[1]),
    "p": nb.pvalues,
}).drop("Intercept")
tab.index = tab.index.str.replace(r"C\(highway_simple\)\[T\.|\]", "", regex=True)
tab["n_segments"] = tab.index.map(seg["highway_simple"].value_counts())

print("Crashes per metre relative to cycleway (the reference class):\n")
print(tab.sort_values("rate_ratio", ascending=False)
      .to_string(float_format=lambda v: f"{v:.3f}"))

sp = nb.params["maxspeed_num"]
print(f"\nmaxspeed: {np.exp(sp):.4f} per km/h")
print(f"  50 km/h vs 30 km/h: {np.exp(sp * 20):.3f}x crashes per metre")

Crashes per metre relative to cycleway (the reference class):

                 coef  rate_ratio  ci_low   ci_high     p  n_segments
primary         7.317    1506.204 209.929 10806.744 0.000    4796.000
secondary       6.773     874.056 121.915  6266.433 0.000   16337.000
tertiary        6.636     762.106 106.299  5463.867 0.000   11897.000
secondary_link  6.110     450.366  60.372  3359.673 0.000     256.000
primary_link    5.974     392.990  51.689  2987.913 0.000     141.000
other           5.951     384.020  10.807 13645.486 0.001       5.000
cycleway        5.715     303.393  40.434  2276.468 0.000   13637.000
residential     5.471     237.808  33.201  1703.358 0.000   70549.000
unclassified    5.257     191.986  26.670  1382.047 0.000    2326.000
trunk           4.931     138.543  10.891  1762.382 0.000      10.000
living_street   4.724     112.670  15.641   811.612 0.000    3589.000
tertiary_link   4.570      96.551   8.047  1158.515 0.000      67.000
pedestrian      4.464      

In [7]:
# Which class becomes the reference is alphabetical by default, and the expanded
# highway_simple now starts with a rare class — which is why the rate ratios came
# out in the hundreds. Setting cycleway explicitly makes them interpretable:
# cycleway is the protected-infrastructure baseline the comparison is about.
FORMULA = ('crash_count ~ C(highway_simple, Treatment(reference="cycleway")) '
           '+ maxspeed_num + has_cycleway')

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    nb = smf.glm(FORMULA, data=seg,
                 family=sm.families.NegativeBinomial(alpha=best_alpha),
                 offset=seg["log_len"]).fit()

tab = pd.DataFrame({
    "rate_ratio": np.exp(nb.params),
    "ci_low": np.exp(nb.conf_int()[0]),
    "ci_high": np.exp(nb.conf_int()[1]),
    "p": nb.pvalues,
}).drop("Intercept")
tab.index = (tab.index
             .str.replace(r'C\(highway_simple, Treatment\(reference="cycleway"\)\)\[T\.', "", regex=True)
             .str.replace(r"\]$", "", regex=True))
tab["n_segments"] = tab.index.map(seg["highway_simple"].value_counts())

# Classes with a handful of segments cannot support an estimate; shown but flagged.
print("Crashes per metre relative to cycleway:\n")
print(tab.sort_values("rate_ratio", ascending=False)
      .to_string(float_format=lambda v: f"{v:.3f}"))

Crashes per metre relative to cycleway:

                rate_ratio  ci_low  ci_high     p  n_segments
primary              4.965   3.189    7.728 0.000    4796.000
secondary            2.881   1.857    4.470 0.000   16337.000
tertiary             2.512   1.619    3.897 0.000   11897.000
secondary_link       1.484   0.821    2.684 0.191     256.000
primary_link         1.295   0.674    2.491 0.438     141.000
other                1.266   0.062   25.683 0.878       5.000
maxspeed_num         1.006   1.003    1.009 0.000         NaN
residential          0.784   0.508    1.210 0.271   70549.000
has_cycleway         0.640   0.416    0.984 0.042         NaN
unclassified         0.633   0.401    0.998 0.049    2326.000
trunk                0.457   0.086    2.420 0.357      10.000
living_street        0.371   0.234    0.588 0.000    3589.000
tertiary_link        0.318   0.066    1.540 0.155      67.000
pedestrian           0.286   0.164    0.499 0.000     612.000
service              0.046   

### 3.1 Crashes per metre by road class

Rate ratios relative to `cycleway`, from the negative binomial with a log-length
offset. Classes with under 300 segments are omitted as uninformative.

| Class | Rate ratio | 95% CI | Segments |
|---|---|---|---|
| `primary` | **4.97** | 3.19–7.73 | 4,796 |
| `secondary` | 2.88 | 1.86–4.47 | 16,337 |
| `tertiary` | 2.51 | 1.62–3.90 | 11,897 |
| `cycleway` | 1.00 | reference | 13,637 |
| `residential` | 0.78 | 0.51–1.21 | 70,549 |
| `living_street` | **0.37** | 0.23–0.59 | 3,589 |
| `pedestrian` | 0.29 | 0.16–0.50 | 612 |
| `service` | 0.05 | 0.03–0.07 | 93,315 |
| `path` | 0.01 | 0.006–0.017 | 13,445 |
| `track` | 0.002 | 0.001–0.005 | 7,641 |

**A residential street is statistically indistinguishable from a cycleway**
(0.78, CI 0.51–1.21, p = 0.271) — on 70,549 segments, so this is a well-powered
null rather than an absent effect.

`maxspeed_num` is 1.006 per km/h [1.003–1.009], so a 50 km/h road carries 13% more
crashes per metre than a 30 km/h road, holding road class constant.

### 3.2 The bottom of the table is exposure, not safety

`service`, `path`, `track` and